<a href="https://colab.research.google.com/github/mostafadentist/healthcare-data-analytics/blob/main/dental_centers_analytical_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate synthetic waiting time data
def generate_waiting_time_data():
    branches = (
        ['Shiny White Branch ' + str(i) for i in range(1, 7)] +
        ['Wonders Dental Branch ' + str(i) for i in range(1, 4)]
    )

    data = []
    for branch in branches:
        # Different centers have different efficiency levels
        base_wait = np.random.randint(15, 35) if 'Shiny White' in branch else np.random.randint(20, 40)

        for month in pd.date_range('2024-01', '2024-12', freq='MS'):
            # Add seasonal variation
            seasonal_factor = 1 + 0.2 * np.sin(month.month * np.pi / 6)

            # Generate daily data for the month
            days_in_month = pd.Period(month, freq='M').days_in_month
            for day in range(days_in_month):
                wait_time = base_wait * seasonal_factor + np.random.normal(0, 5)
                wait_time = max(5, wait_time)  # Minimum 5 minutes

                data.append({
                    'Branch': branch,
                    'Date': month + timedelta(days=day),
                    'Avg_Waiting_Time_Minutes': round(wait_time, 1),
                    'Patients_Seen': np.random.randint(15, 45)
                })

    return pd.DataFrame(data)

# Generate data
df_waiting = generate_waiting_time_data()

# Calculate monthly averages
monthly_avg = df_waiting.groupby([pd.Grouper(key='Date', freq='M'), 'Branch'])['Avg_Waiting_Time_Minutes'].mean().reset_index()

# Create interactive plot
fig = px.line(monthly_avg,
              x='Date',
              y='Avg_Waiting_Time_Minutes',
              color='Branch',
              title='Average Patient Waiting Time by Branch (2024)',
              labels={'Avg_Waiting_Time_Minutes': 'Average Waiting Time (minutes)',
                      'Date': 'Month'},
              line_shape='spline')

fig.update_layout(
    hovermode='x unified',
    height=500,
    template='plotly_white',
    font=dict(size=12),
    title_font_size=16,
    showlegend=True,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02)
)

fig.add_hline(y=30, line_dash="dash", line_color="red",
              annotation_text="Target: 30 min", annotation_position="left")

fig.show()

print(f"Dataset shape: {df_waiting.shape}")
print(f"Average waiting time across all branches: {df_waiting['Avg_Waiting_Time_Minutes'].mean():.1f} minutes")

/tmp/ipython-input-1998062932.py:47: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_avg = df_waiting.groupby([pd.Grouper(key='Date', freq='M'), 'Branch'])['Avg_Waiting_Time_Minutes'].mean().reset_index()


Dataset shape: (3294, 4)
Average waiting time across all branches: 24.8 minutes


In [3]:
# Generate synthetic implant procedure data
def generate_implant_data():
    branches = (
        ['Shiny White Branch ' + str(i) for i in range(1, 7)] +
        ['Wonders Dental Branch ' + str(i) for i in range(1, 4)]
    )

    data = []

    for branch in branches:
        # Each branch has different implant capacity
        monthly_capacity = np.random.randint(20, 50) if 'Shiny White' in branch else np.random.randint(15, 35)

        for month in pd.date_range('2023-01', '2024-12', freq='MS'):
            # Generate implant procedures with success rates
            procedures = int(monthly_capacity * np.random.uniform(0.6, 1.0))
            success_rate = np.random.uniform(0.94, 0.99)  # High success rate typical for implants

            data.append({
                'Branch': branch,
                'Month': month,
                'Total_Implants': procedures,
                'Successful_Implants': int(procedures * success_rate),
                'Revenue_USD': procedures * np.random.uniform(1800, 2500),
                'Avg_Procedure_Time_Hours': np.random.uniform(1.5, 3.0)
            })

    return pd.DataFrame(data)

# Generate data
df_implants = generate_implant_data()
df_implants['Success_Rate'] = (df_implants['Successful_Implants'] / df_implants['Total_Implants'] * 100).round(1)

# Group by dental center
df_implants['Center'] = df_implants['Branch'].apply(lambda x: 'Shiny White' if 'Shiny White' in x else 'Wonders Dental')

# Create subplot figure
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Total Implants by Center', 'Success Rate Trend',
                    'Revenue from Implants', 'Implants Distribution by Branch'),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'bar'}, {'type': 'pie'}]]
)

# Subplot 1: Total implants by center
center_totals = df_implants.groupby(['Month', 'Center'])['Total_Implants'].sum().reset_index()
for center in ['Shiny White', 'Wonders Dental']:
    center_data = center_totals[center_totals['Center'] == center]
    fig.add_trace(
        go.Bar(name=center, x=center_data['Month'], y=center_data['Total_Implants']),
        row=1, col=1
    )

# Subplot 2: Success rate trend
avg_success = df_implants.groupby('Month')['Success_Rate'].mean().reset_index()
fig.add_trace(
    go.Scatter(x=avg_success['Month'], y=avg_success['Success_Rate'],
               mode='lines+markers', name='Success Rate', line=dict(color='green')),
    row=1, col=2
)

# Subplot 3: Revenue
revenue_by_center = df_implants.groupby('Center')['Revenue_USD'].sum().reset_index()
fig.add_trace(
    go.Bar(x=revenue_by_center['Center'], y=revenue_by_center['Revenue_USD'],
           marker_color=['#1f77b4', '#ff7f0e'], showlegend=False),
    row=2, col=1
)

# Subplot 4: Branch distribution (2024 only)
branch_2024 = df_implants[df_implants['Month'].dt.year == 2024].groupby('Branch')['Total_Implants'].sum()
fig.add_trace(
    go.Pie(labels=branch_2024.index, values=branch_2024.values, hole=0.3),
    row=2, col=2
)

fig.update_layout(height=700, showlegend=True, title_text="Dental Implants KPI Dashboard (2023-2024)")
fig.update_xaxes(title_text="Month", row=1, col=1)
fig.update_xaxes(title_text="Month", row=1, col=2)
fig.update_xaxes(title_text="Dental Center", row=2, col=1)
fig.update_yaxes(title_text="Number of Implants", row=1, col=1)
fig.update_yaxes(title_text="Success Rate (%)", row=1, col=2)
fig.update_yaxes(title_text="Revenue (USD)", row=2, col=1)

fig.show()

print(f"Total implants performed: {df_implants['Total_Implants'].sum()}")
print(f"Average success rate: {df_implants['Success_Rate'].mean():.1f}%")
print(f"Total revenue from implants: ${df_implants['Revenue_USD'].sum():,.2f}")

Total implants performed: 4511
Average success rate: 93.9%
Total revenue from implants: $9,705,434.92


In [4]:
# Generate orthodontic treatment data
def generate_ortho_data():
    branches = (
        ['Shiny White Branch ' + str(i) for i in range(1, 7)] +
        ['Wonders Dental Branch ' + str(i) for i in range(1, 4)]
    )

    treatment_types = ['Traditional Braces', 'Clear Aligners', 'Ceramic Braces', 'Lingual Braces']
    age_groups = ['Children (7-12)', 'Teens (13-18)', 'Adults (19-35)', 'Adults (36+)']

    data = []

    for branch in branches:
        for month in pd.date_range('2024-01', '2024-12', freq='MS'):
            for treatment in treatment_types:
                for age_group in age_groups:
                    # Different treatment preferences by age
                    if 'Clear Aligners' in treatment and 'Adults' in age_group:
                        base_patients = np.random.randint(5, 15)
                    elif 'Traditional' in treatment and 'Children' in age_group:
                        base_patients = np.random.randint(8, 20)
                    else:
                        base_patients = np.random.randint(2, 10)

                    data.append({
                        'Branch': branch,
                        'Month': month,
                        'Treatment_Type': treatment,
                        'Age_Group': age_group,
                        'New_Patients': base_patients,
                        'Active_Patients': base_patients * np.random.randint(10, 25),
                        'Completed_Treatments': np.random.randint(0, base_patients),
                        'Avg_Treatment_Duration_Months': np.random.uniform(12, 36)
                    })

    return pd.DataFrame(data)

# Generate data
df_ortho = generate_ortho_data()

# Aggregate by treatment type
treatment_summary = df_ortho.groupby('Treatment_Type').agg({
    'New_Patients': 'sum',
    'Active_Patients': 'mean',
    'Completed_Treatments': 'sum'
}).round(0).reset_index()

# Create visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Orthodontic Treatments by Type', 'Age Group Distribution',
                    'Monthly New Patient Trend', 'Treatment Completion Rate'),
    specs=[[{'type': 'bar'}, {'type': 'sunburst'}],
           [{'type': 'scatter'}, {'type': 'indicator'}]]
)

# Subplot 1: Treatment types
fig.add_trace(
    go.Bar(x=treatment_summary['Treatment_Type'],
           y=treatment_summary['New_Patients'],
           marker_color='lightblue',
           text=treatment_summary['New_Patients'],
           textposition='auto'),
    row=1, col=1
)

# Subplot 2: Age group sunburst
age_data = df_ortho.groupby(['Age_Group', 'Treatment_Type'])['New_Patients'].sum().reset_index()
fig.add_trace(
    go.Sunburst(
        labels=age_data['Age_Group'].tolist() + age_data['Treatment_Type'].tolist(),
        parents=[''] * len(age_data['Age_Group']) + age_data['Age_Group'].tolist(),
        values=age_data['New_Patients'].tolist() + age_data['New_Patients'].tolist()
    ),
    row=1, col=2
)

# Subplot 3: Monthly trend
monthly_trend = df_ortho.groupby('Month')['New_Patients'].sum().reset_index()
fig.add_trace(
    go.Scatter(x=monthly_trend['Month'], y=monthly_trend['New_Patients'],
               mode='lines+markers', line=dict(color='purple', width=2),
               marker=dict(size=8)),
    row=2, col=1
)

# Subplot 4: Completion rate indicator
completion_rate = (df_ortho['Completed_Treatments'].sum() / df_ortho['New_Patients'].sum() * 100)
fig.add_trace(
    go.Indicator(
        mode="gauge+number+delta",
        value=completion_rate,
        title={'text': "Completion Rate (%)"},
        delta={'reference': 75},
        gauge={'axis': {'range': [None, 100]},
               'bar': {'color': "darkblue"},
               'steps': [
                   {'range': [0, 50], 'color': "lightgray"},
                   {'range': [50, 80], 'color': "gray"}],
               'threshold': {'line': {'color': "red", 'width': 4},
                           'thickness': 0.75, 'value': 90}}
    ),
    row=2, col=2
)

fig.update_layout(height=700, title_text="Orthodontic Treatment Analytics Dashboard")
fig.show()

print(f"Total new orthodontic patients in 2024: {df_ortho['New_Patients'].sum()}")
print(f"Most popular treatment: {treatment_summary.loc[treatment_summary['New_Patients'].idxmax(), 'Treatment_Type']}")

Total new orthodontic patients in 2024: 11265
Most popular treatment: Clear Aligners


In [5]:
# Generate comprehensive revenue data
def generate_revenue_data():
    branches = (
        ['Shiny White Branch ' + str(i) for i in range(1, 7)] +
        ['Wonders Dental Branch ' + str(i) for i in range(1, 4)]
    )

    service_categories = {
        'General Dentistry': (50, 200),
        'Cosmetic Dentistry': (200, 1000),
        'Orthodontics': (2000, 6000),
        'Oral Surgery': (500, 3000),
        'Periodontics': (150, 800),
        'Endodontics': (300, 1200),
        'Preventive Care': (30, 150)
    }

    data = []

    for branch in branches:
        branch_multiplier = 1.2 if 'Shiny White' in branch else 1.0

        for month in pd.date_range('2023-01', '2024-12', freq='MS'):
            for service, (min_price, max_price) in service_categories.items():
                # Seasonal adjustments
                seasonal_factor = 1 + 0.15 * np.sin((month.month - 3) * np.pi / 6)

                num_procedures = np.random.randint(10, 100)
                avg_price = np.random.uniform(min_price, max_price) * branch_multiplier * seasonal_factor

                revenue = num_procedures * avg_price
                costs = revenue * np.random.uniform(0.3, 0.5)  # 30-50% costs

                data.append({
                    'Branch': branch,
                    'Month': month,
                    'Service_Category': service,
                    'Revenue': revenue,
                    'Costs': costs,
                    'Profit': revenue - costs,
                    'Procedures': num_procedures,
                    'Avg_Price': avg_price
                })

    return pd.DataFrame(data)

# Generate data
df_revenue = generate_revenue_data()
df_revenue['Profit_Margin'] = (df_revenue['Profit'] / df_revenue['Revenue'] * 100).round(1)
df_revenue['Center'] = df_revenue['Branch'].apply(lambda x: 'Shiny White' if 'Shiny White' in x else 'Wonders Dental')

# Create comprehensive revenue dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=('Monthly Revenue Trend', 'Revenue by Service Category',
                    'Profit Margin by Center', 'Top Performing Branches',
                    'Revenue Growth Rate', 'Service Mix'),
    specs=[[{'type': 'scatter'}, {'type': 'bar'}],
           [{'type': 'box'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'pie'}]],
    vertical_spacing=0.1
)

# 1. Monthly revenue trend
monthly_revenue = df_revenue.groupby(['Month', 'Center'])['Revenue'].sum().reset_index()
for center in ['Shiny White', 'Wonders Dental']:
    center_data = monthly_revenue[monthly_revenue['Center'] == center]
    fig.add_trace(
        go.Scatter(x=center_data['Month'], y=center_data['Revenue'],
                   name=center, mode='lines+markers'),
        row=1, col=1
    )

# 2. Revenue by service category
service_revenue = df_revenue.groupby('Service_Category')['Revenue'].sum().sort_values(ascending=True)
fig.add_trace(
    go.Bar(x=service_revenue.values, y=service_revenue.index,
           orientation='h', marker_color='teal'),
    row=1, col=2
)

# 3. Profit margin distribution
fig.add_trace(
    go.Box(x=df_revenue['Center'], y=df_revenue['Profit_Margin'],
           marker_color='orange'),
    row=2, col=1
)

# 4. Top performing branches (2024 only)
branch_performance = df_revenue[df_revenue['Month'].dt.year == 2024].groupby('Branch')['Revenue'].sum().sort_values(ascending=False).head(5)
fig.add_trace(
    go.Bar(x=branch_performance.index, y=branch_performance.values,
           marker_color='purple'),
    row=2, col=2
)

# 5. Revenue growth rate
monthly_total = df_revenue.groupby('Month')['Revenue'].sum().reset_index()
monthly_total['Growth_Rate'] = monthly_total['Revenue'].pct_change() * 100
fig.add_trace(
    go.Scatter(x=monthly_total['Month'], y=monthly_total['Growth_Rate'],
               mode='lines+markers', line=dict(color='red')),
    row=3, col=1
)

# 6. Service mix pie chart
service_mix = df_revenue.groupby('Service_Category')['Revenue'].sum()
fig.add_trace(
    go.Pie(labels=service_mix.index, values=service_mix.values, hole=0.4),
    row=3, col=2
)

fig.update_layout(height=900, showlegend=True, title_text="Comprehensive Revenue Analytics Dashboard")
fig.show()

# Summary statistics
print(f"Total Revenue (2023-2024): ${df_revenue['Revenue'].sum():,.2f}")
print(f"Average Profit Margin: {df_revenue['Profit_Margin'].mean():.1f}%")
print(f"Best Performing Service: {service_revenue.idxmax()}")

Total Revenue (2023-2024): $97,588,777.79
Average Profit Margin: 59.9%
Best Performing Service: Orthodontics


In [6]:
# Generate claims data
def generate_claims_data():
    branches = (
        ['Shiny White Branch ' + str(i) for i in range(1, 7)] +
        ['Wonders Dental Branch ' + str(i) for i in range(1, 4)]
    )

    rejection_reasons = [
        'Missing Documentation', 'Invalid Procedure Code', 'Eligibility Issues',
        'Prior Authorization Required', 'Duplicate Claim', 'Timely Filing',
        'Incorrect Patient Info', 'Service Not Covered'
    ]

    insurance_providers = ['BlueCross', 'Aetna', 'Cigna', 'Delta Dental', 'MetLife', 'United Healthcare']

    data = []

    for branch in branches:
        for month in pd.date_range('2024-01', '2024-12', freq='MS'):
            total_claims = np.random.randint(100, 300)

            for provider in insurance_providers:
                provider_claims = int(total_claims * np.random.uniform(0.1, 0.25))
                rejection_rate = np.random.uniform(0.05, 0.20)  # 5-20% rejection rate
                rejected_claims = int(provider_claims * rejection_rate)

                for _ in range(rejected_claims):
                    reason = np.random.choice(rejection_reasons)
                    claim_amount = np.random.uniform(100, 2000)

                    data.append({
                        'Branch': branch,
                        'Month': month,
                        'Insurance_Provider': provider,
                        'Rejection_Reason': reason,
                        'Claim_Amount': claim_amount,
                        'Days_to_Resolution': np.random.randint(5, 45),
                        'Resolved': np.random.choice([True, False], p=[0.8, 0.2])
                    })

                # Add accepted claims summary
                data.append({
                    'Branch': branch,
                    'Month': month,
                    'Insurance_Provider': provider,
                    'Rejection_Reason': 'Accepted',
                    'Claim_Amount': (provider_claims - rejected_claims) * np.random.uniform(100, 500),
                    'Days_to_Resolution': 0,
                    'Resolved': True
                })

    return pd.DataFrame(data)

# Generate data
df_claims = generate_claims_data()

# Calculate metrics
rejection_by_reason = df_claims[df_claims['Rejection_Reason'] != 'Accepted'].groupby('Rejection_Reason').size().sort_values(ascending=False)
rejection_by_provider = df_claims[df_claims['Rejection_Reason'] != 'Accepted'].groupby('Insurance_Provider').agg({
    'Claim_Amount': 'sum',
    'Branch': 'count'
}).rename(columns={'Branch': 'Rejection_Count'})

# Create visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top Rejection Reasons', 'Rejection Trend Over Time',
                    'Financial Impact by Provider', 'Resolution Time Distribution'),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'bar'}, {'type': 'histogram'}]]
)

# 1. Top rejection reasons
fig.add_trace(
    go.Bar(x=rejection_by_reason.values, y=rejection_by_reason.index,
           orientation='h', marker_color='red', text=rejection_by_reason.values,
           textposition='auto'),
    row=1, col=1
)

# 2. Rejection trend
monthly_rejections = df_claims[df_claims['Rejection_Reason'] != 'Accepted'].groupby('Month').size().reset_index(name='Rejections')
fig.add_trace(
    go.Scatter(x=monthly_rejections['Month'], y=monthly_rejections['Rejections'],
               mode='lines+markers', line=dict(color='darkred', width=2)),
    row=1, col=2
)

# 3. Financial impact
fig.add_trace(
    go.Bar(x=rejection_by_provider.index, y=rejection_by_provider['Claim_Amount'],
           marker_color='orange', text=rejection_by_provider['Claim_Amount'].round(0),
           textposition='auto'),
    row=2, col=1
)

# 4. Resolution time
resolution_data = df_claims[df_claims['Rejection_Reason'] != 'Accepted']['Days_to_Resolution']
fig.add_trace(
    go.Histogram(x=resolution_data, nbinsx=20, marker_color='blue'),
    row=2, col=2
)

fig.update_layout(height=700, showlegend=False, title_text="Insurance Claims Rejection Analysis Dashboard")
fig.update_xaxes(title_text="Number of Rejections", row=1, col=1)
fig.update_xaxes(title_text="Month", row=1, col=2)
fig.update_xaxes(title_text="Insurance Provider", row=2, col=1)
fig.update_xaxes(title_text="Days to Resolution", row=2, col=2)
fig.update_yaxes(title_text="Rejection Reason", row=1, col=1)
fig.update_yaxes(title_text="Number of Rejections", row=1, col=2)
fig.update_yaxes(title_text="Total Claim Amount ($)", row=2, col=1)
fig.update_yaxes(title_text="Frequency", row=2, col=2)

fig.show()

# Calculate key metrics
total_rejections = len(df_claims[df_claims['Rejection_Reason'] != 'Accepted'])
total_claims = len(df_claims)
rejection_rate = (total_rejections / total_claims) * 100

print(f"Total Claims Processed: {total_claims}")
print(f"Total Rejections: {total_rejections}")
print(f"Overall Rejection Rate: {rejection_rate:.1f}%")
print(f"Average Days to Resolution: {df_claims[df_claims['Rejection_Reason'] != 'Accepted']['Days_to_Resolution'].mean():.1f}")
print(f"Total Financial Impact of Rejections: ${df_claims[df_claims['Rejection_Reason'] != 'Accepted']['Claim_Amount'].sum():,.2f}")

Total Claims Processed: 2989
Total Rejections: 2341
Overall Rejection Rate: 78.3%
Average Days to Resolution: 24.4
Total Financial Impact of Rejections: $2,531,860.96
